# Text Splitter — 청킹 (Chunking)


### 왜 분할이 필요한가?

| 문제 | 해결 |
|------|------|
| LLM 컨텍스트 윈도우 제한 | 작은 청크로 분할 |
| 긴 문서에서 정확한 검색 어려움 | 관련 청크만 검색 |
| 토큰 비용 | 효율적인 토큰 사용 |


### 참고자료
- https://docs.langchain.com/oss/python/integrations/splitters
- https://wikidocs.net/231430

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-community langchain-text-splitters langchain-openai langchain-experimental pypdfium2 pypdf

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. 데모용 긴 문서

In [2]:
SAMPLE_DOC = """
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.

### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.

### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.
"""

print(f"문서 길이: {len(SAMPLE_DOC)} 자")

문서 길이: 971 자


## 3. `RecursiveCharacterTextSplitter`, 가장 많이 쓰는 분할기

In [4]:
# RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,             # 한 청크 목표 크기 (자)
    chunk_overlap=60,           # 겹치는 글자 수 (맥락 보존)
    separators=['\n\n', '\n', '다.', ' ', '']     # 분할 우선순위
)

chunks = splitter.split_text(SAMPLE_DOC)
print(f'청크 수: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'---청크 {i} ({len(c)}자) ---')
    print((c[:150]))
    print()

청크 수: 4
---청크 0 (244자) ---
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션

---청크 1 (241자) ---
### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에

---청크 2 (238자) ---
### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의

---청크 3 (240자) ---
### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야



## 4. overlap 효과,
- 왜 80자를 겹치게 두나?
- 청크 경계에서 문장이 끊기면 의미가 잘림
- 인접 청크끼리 일부를 겹치게 두면 검색 시 한 청크가 잡혀도 양쪽 맥락이 살아남

### overlap 0 일 때

In [5]:
splitter_no_overlap = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
)

chunks_no_ovl = splitter_no_overlap.split_text(SAMPLE_DOC)

# 각 청크의 마지막 / 다음 청크의 처음 비교
# overlap이 없으면 앞 청크의 끝과 다음 청크의 시작이 이어지지만, 같은 내용 반복x
for i in range(len(chunks_no_ovl)-1):
    end = chunks_no_ovl[i][-30:]
    start = chunks_no_ovl[i+1][30:]
    print(f'청크{i} 끝 : ...{end!r}')
    print(f'청크{i+1} 시작 : {start!r}...')
    print()

청크0 끝 : ...'스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.'
청크1 시작 : ' 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'...

청크1 끝 : ...'무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'
청크2 시작 : '은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'...

청크2 끝 : ...'첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'
청크3 시작 : '문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.'...



In [8]:
# 각 청크의 마지막 / 다음 청크의 처음 비교
for i in range(len(chunks_no_ovl) - 1):
    end = chunks_no_ovl[i][-30:]
    start = chunks_no_ovl[i + 1][:30]
    print(f"청크{i} 끝: ...{end!r}")
    print(f"청크{i+1} 시작: {start!r}...")
    print()

청크0 끝: ...'스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.'
청크1 시작: '### 2. 재택근무 신청\n재택근무는 최소 하루 전까지'...

청크1 끝: ...'무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'
청크2 시작: '### 3. 법인카드 및 경비 처리\n법인카드 사용 내역'...

청크2 끝: ...'첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'
청크3 시작: '### 4. 보안 및 개인정보 보호\n개인정보가 포함된 '...



## 5. Token 기준 분할

- LLM 컨텍스트는 "자" 가 아니라 "토큰" 으로 제한됨
- 토큰 기준으로 자르려면:

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='text-embedding-3-small',
    chunk_size=200,
    chunk_overlap=40
)

In [7]:

token_chunks = token_splitter.split_text(SAMPLE_DOC)
print(f"토큰 기준 청크 수: {len(token_chunks)}")
for i, c in enumerate(token_chunks):
    print(f"--- 청크 {i} ({len(c)}자) ---")
    print(c[:120])
    print()

토큰 기준 청크 수: 10
--- 청크 0 (15자) ---
## 회사 업무 운영 가이드

--- 청크 1 (17자) ---
### 1. 신규 입사자 온보딩

--- 청크 2 (209자) ---
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 

--- 청크 3 (14자) ---
### 2. 재택근무 신청

--- 청크 4 (190자) ---
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근

--- 청크 5 (70자) ---
10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

--- 청크 6 (19자) ---
### 3. 법인카드 및 경비 처리

--- 청크 7 (218자) ---
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 

--- 청크 8 (19자) ---
### 4. 보안 및 개인정보 보호

--- 청크 9 (220자) ---
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공



## 6. 청크 크기 trade-off

| 청크 크기 | 장점 | 단점 |
|---|---|---|
| 작음 (200자) | 검색 정밀도 ↑, 토큰 비용 ↓ | 맥락 부족, 청크 수 ↑ |
| 중간 (500~800자) | 균형 | (없음) |
| 큼 (1500자+) | 풍부한 맥락 | 검색 정밀도 ↓, 비용 ↑ |

실무 시작점은 문서 성격에 따라 다르지만, 일반 문서 RAG 는 보통 300 ~ 800 토큰, overlap 10 ~ 20% 정도에서 실험을 시작합니다.


## 7. 정리

- 긴 문서는 분할 필수
- `RecursiveCharacterTextSplitter` 가 기본
- `chunk_overlap` 으로 경계 손실 방지
- 토큰 기준 분할은 `from_tiktoken_encoder`


## [실습]
1. `chunk_size` 를 100 / 800 / 2000 으로 바꿔 청크 수 변화.
2. `separators` 를 한국어 친화로 (`["\n\n", "\n", ". ", "다. ", " "]`) 바꿔 분할 결과 비교.
3. PDF (`PyPDFLoader` 사용) 를 로드해 분할.
4. 마크다운 헤더 기준 분할 (`MarkdownHeaderTextSplitter`) 시도, 청크가 의미 단위로 떨어지는지.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='text-embedding-3-small',
    chunk_size=100,
    chunk_overlap=40,
    separators=['\n\n', '\n', '다.', ' ', '']     # 분할 우선순위
)

from langchain_community.document_loaders import PyPDFium2Loader

filename = 'data2/BOK 이슈노트 제2022-38호 인공지능 언어모형을 이용한 인플레이션 어조지수 개발 및 시사점.pdf'
loader2 = PyPDFium2Loader(filename)
data = loader2.load()

token_chunks = token_splitter.split_text(data[5].page_content)
print(f"토큰 기준 청크 수: {len(token_chunks)}")
for i, c in enumerate(token_chunks):
    print(f"--- 청크 {i} ({len(c)}자) ---")
    print(c[:120])
    print()

토큰 기준 청크 수: 32
--- 청크 0 (92자) ---
제 2022-38호
6 한국은행
그러나, 아직 기존 텍스트 분석기법에 비해 인
공지능 언어모형은 구조와 분석방법이 복잡하
여 경제분석에 활용하기 어려운 것으로 평가되

--- 청크 1 (109자) ---
여 경제분석에 활용하기 어려운 것으로 평가되
고 있다(Shapiro et al., 2017). 실제 중앙은
행을 비롯한 대부분 경제 분석에서는 아직 기
존 텍스트 분석기법을 이용한 연구가 큰 비중

--- 청크 2 (93자) ---
존 텍스트 분석기법을 이용한 연구가 큰 비중
을 차지하고 있다. 본 연구는 효율적인 어조분
류 모형 개발과 향후 활용성을 고려하여 인공
지능 언어모형을 이용하였다.9)

--- 청크 3 (72자) ---
지능 언어모형을 이용하였다.9)
2. 미세조정
사전훈련된 언어모형을 미세조정하여 뉴스
기사에 나타난 인플레이션 어조를 측정하는 절

--- 청크 4 (75자) ---
기사에 나타난 인플레이션 어조를 측정하는 절
차는 다음과 같다. 미세조정을 위한 학습데이
터는 네이버 뉴스에서 물가 관련 키워드10)로

--- 청크 5 (78자) ---
터는 네이버 뉴스에서 물가 관련 키워드10)로 
검색하여 수집한 뉴스기사를 이용하였다. 데
이터는 2002년 2월부터 2022년 6월까지 총

--- 청크 6 (84자) ---
이터는 2002년 2월부터 2022년 6월까지 총 
188만건(일평균 418건)의 뉴스 기사에서 수
집된 6,406만개(일평균 8,653개) 문장이다.

--- 청크 7 (85자) ---
집된 6,406만개(일평균 8,653개) 문장이다. 
학습데이터는 전체 문장 중 5,000개를 임의로 
추출한 다음 아래 <표 2>와 같이 대상품목, 현

--- 청크 8 (80자) ---
추출한 다음 아래 <표 2>와 같이 대상품목, 현
재 어조, 미래 어조 등 3개 카테고리에 대해 라
벨을 지정하여 구축하였다. 대상품목은 식품,

--- 청크 9 (75자) ---
벨을 지정하여 

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter2 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='text-embedding-3-small',
    chunk_size=800,
    chunk_overlap=40,
    separators=['\n\n', '\n', '다.', ' ', '']     # 분할 우선순위
)

from langchain_community.document_loaders import PyPDFium2Loader

filename = 'data2/BOK 이슈노트 제2022-38호 인공지능 언어모형을 이용한 인플레이션 어조지수 개발 및 시사점.pdf'
loader3 = PyPDFium2Loader(filename)
data = loader3.load()

token_chunks = token_splitter2.split_text(data[5].page_content)
print(f"토큰 기준 청크 수: {len(token_chunks)}")
for i, c in enumerate(token_chunks):
    print(f"--- 청크 {i} ({len(c)}자) ---")
    print(c[:120])
    print()

토큰 기준 청크 수: 3
--- 청크 0 (776자) ---
제 2022-38호
6 한국은행
그러나, 아직 기존 텍스트 분석기법에 비해 인
공지능 언어모형은 구조와 분석방법이 복잡하
여 경제분석에 활용하기 어려운 것으로 평가되
고 있다(Shapiro et al., 2017).

--- 청크 1 (832자) ---
미래 어조)으로 미세조정하는 데 이용된다.
〈표 2〉 인플레이션 어조분류 학습데이터(예시)
문장 주제1)
어조2)
현재 미래
휘발유 가격, 1400원대로 떨어졌다. 
다음 주도 하락 전망 8 1 1
이에따라 주가상승

--- 청크 2 (311자) ---
10) 물가, 가격, 값, 인플레, 디플레, 비용, 부담, 요금 등 총 8개의 물가 관련 키워드를 기준으로 검색하였다.
11) 세부 분류기준 및 분류 결과는 <부록 1>을 참조하라.
12) ‘다만 근원물가 상승세가 



In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter3 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='text-embedding-3-small',
    chunk_size=2000,
    chunk_overlap=40,
    separators=['\n\n', '\n', '다.', ' ', '']     # 분할 우선순위
)

from langchain_community.document_loaders import PyPDFium2Loader

filename = 'data2/BOK 이슈노트 제2022-38호 인공지능 언어모형을 이용한 인플레이션 어조지수 개발 및 시사점.pdf'
loader4 = PyPDFium2Loader(filename)
data = loader4.load()

token_chunks = token_splitter3.split_text(data[5].page_content)
print(f"토큰 기준 청크 수: {len(token_chunks)}")
for i, c in enumerate(token_chunks):
    print(f"--- 청크 {i} ({len(c)}자) ---")
    print(c[:120])
    print()

토큰 기준 청크 수: 1
--- 청크 0 (1897자) ---
제 2022-38호
6 한국은행
그러나, 아직 기존 텍스트 분석기법에 비해 인
공지능 언어모형은 구조와 분석방법이 복잡하
여 경제분석에 활용하기 어려운 것으로 평가되
고 있다(Shapiro et al., 2017).

